# Deep Research Agentic App
**Framework:** OpenAI Agents SDK (`openai-agents`)  
**Architecture:** Multi-agent pipeline — Planner → Researcher → Synthesizer → Writer  
**Search:** Tavily Search API (real-time web retrieval)  
**LLM:** OpenRouter (frontier models via OpenAI-compatible interface)  
**Output:** Structured Markdown research report with citations

In [ ]:
# Cell 1 — Install dependencies
%pip install openai-agents tavily-python python-dotenv pydantic gradio --quiet

In [ ]:
# Cell 2 — Imports
import asyncio
import json
import os
import textwrap
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from tavily import TavilyClient

from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    RunConfig,
    Runner,
    function_tool,
    handoff,
)

In [ ]:
# Cell 3 — Constants
load_dotenv(override=True)

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
TAVILY_API_KEY     = os.environ["TAVILY_API_KEY"]

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Primary reasoning model — strong, fast, cost-effective
PRIMARY_MODEL = "google/gemini-2.5-flash"

# Max search results per query
MAX_SEARCH_RESULTS = 5

# Max sub-questions the planner will generate
MAX_SUB_QUESTIONS  = 5

# Output directory for saved reports
REPORTS_DIR = Path("research_reports")
REPORTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Cell 4 — OpenRouter client + shared model

openrouter_client = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

def get_model() -> OpenAIChatCompletionsModel:
    """Return a configured Chat Completions model pointed at OpenRouter."""
    return OpenAIChatCompletionsModel(
        model=PRIMARY_MODEL,
        openai_client=openrouter_client,
    )

In [ ]:
# Cell 5 — Pydantic schemas for structured agent outputs

class ResearchPlan(BaseModel):
    """Output from the Planner agent."""
    topic: str
    sub_questions: list[str]
    search_queries: list[str]


class SearchResult(BaseModel):
    """A single search result with source attribution."""
    title: str
    url: str
    snippet: str


class ResearchFindings(BaseModel):
    """Output from the Researcher agent."""
    topic: str
    sub_question: str
    findings: str
    sources: list[SearchResult]

In [ ]:
# Cell 6 — Tavily web search tool

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@function_tool
def web_search(query: str) -> str:
    """
    Search the web for current information on a given query.
    Returns a JSON string with a list of results, each containing
    title, url, and content snippet.
    """
    response = tavily_client.search(
        query=query,
        max_results=MAX_SEARCH_RESULTS,
        search_depth="advanced",
    )
    results = [
        {
            "title":   r.get("title", ""),
            "url":     r.get("url", ""),
            "snippet": r.get("content", "")[:600],  # keep tokens in check
        }
        for r in response.get("results", [])
    ]
    return json.dumps(results, ensure_ascii=False)

In [ ]:
# Cell 7 — Agent definitions

# --- Planner Agent ---
# Decomposes the user topic into focused sub-questions and search queries.

planner_agent = Agent(
    name="Planner",
    model=get_model(),
    instructions=textwrap.dedent(f"""
        You are a research planning expert. Given a research topic, break it down
        into at most {MAX_SUB_QUESTIONS} focused sub-questions that together cover
        the topic comprehensively. For each sub-question, also provide one precise
        web search query that would best answer it.

        Respond ONLY with a valid JSON object matching this schema:
        {{
          "topic": "<original topic>",
          "sub_questions": ["<question 1>", ...],
          "search_queries": ["<query 1>", ...]
        }}
        The lists must be the same length. No markdown, no explanation.
    """).strip(),
    output_type=ResearchPlan,
)


# --- Researcher Agent ---
# Takes a sub-question + search query, runs web searches, and summarises findings.

researcher_agent = Agent(
    name="Researcher",
    model=get_model(),
    tools=[web_search],
    instructions=textwrap.dedent("""
        You are a meticulous research analyst. You receive a sub-question and a
        search query. Use the web_search tool to find relevant, current information.
        You may run multiple searches if needed to gather sufficient evidence.

        After researching, return a JSON object matching this schema:
        {
          "topic": "<overall topic>",
          "sub_question": "<the sub-question you answered>",
          "findings": "<a thorough 2-4 paragraph summary of what you found>",
          "sources": [
            {"title": "...", "url": "...", "snippet": "..."},
            ...
          ]
        }
        Be factual, cite evidence from search results. No markdown in output.
    """).strip(),
    output_type=ResearchFindings,
)


# --- Writer Agent ---
# Synthesises all findings into a polished, cited Markdown research report.

writer_agent = Agent(
    name="Writer",
    model=get_model(),
    instructions=textwrap.dedent("""
        You are a professional research writer. You receive a collection of research
        findings (each covering a sub-question) and must compile them into a single,
        well-structured Markdown research report.

        Report structure:
        # <Title>
        **Date:** <today's date>  **Topic:** <topic>

        ## Executive Summary
        (3-5 sentences covering the most important takeaways)

        ## [Section per sub-question]
        (Detailed prose drawing on the findings. Where you reference a source,
        add an inline citation like [Source Title](url).)

        ## Key Takeaways
        (Bullet list of 5-8 concise insights)

        ## References
        (Numbered list of all sources used: [N] Title — URL)

        Write clearly and professionally. Do not invent facts not present in the
        findings. Maintain academic tone throughout.
    """).strip(),
)

In [ ]:
# Cell 8 — Research pipeline orchestration

async def run_research_pipeline(topic: str) -> str:
    """
    Orchestrates the full multi-agent research pipeline:
      1. Planner   — decomposes topic into sub-questions
      2. Researcher — answers each sub-question via web search (parallelised)
      3. Writer    — synthesises all findings into a Markdown report

    Returns the final Markdown report as a string.
    """
    print(f"\n[1/3] Planning research for: {topic!r}")

    # Step 1: Plan
    plan_result = await Runner.run(planner_agent, topic)
    plan: ResearchPlan = plan_result.final_output

    print(f"      Generated {len(plan.sub_questions)} sub-questions:")
    for i, q in enumerate(plan.sub_questions, 1):
        print(f"      {i}. {q}")

    # Step 2: Research each sub-question in parallel
    print(f"\n[2/3] Researching {len(plan.sub_questions)} sub-questions (parallel)...")

    async def research_one(sub_q: str, search_q: str) -> ResearchFindings:
        prompt = (
            f"Overall topic: {plan.topic}\n"
            f"Sub-question: {sub_q}\n"
            f"Start with this search query: {search_q}"
        )
        result = await Runner.run(researcher_agent, prompt)
        return result.final_output

    research_tasks = [
        research_one(sub_q, search_q)
        for sub_q, search_q in zip(plan.sub_questions, plan.search_queries)
    ]
    all_findings: list[ResearchFindings] = await asyncio.gather(*research_tasks)

    print(f"      Completed research on {len(all_findings)} sub-questions.")

    # Step 3: Write the final report
    print("\n[3/3] Synthesising final report...")

    findings_payload = json.dumps(
        [
            {
                "sub_question": f.sub_question,
                "findings":     f.findings,
                "sources":      [s.model_dump() for s in f.sources],
            }
            for f in all_findings
        ],
        ensure_ascii=False,
        indent=2,
    )

    writer_prompt = (
        f"Topic: {plan.topic}\n"
        f"Date: {datetime.now().strftime('%B %d, %Y')}\n\n"
        f"Research findings (JSON):\n{findings_payload}"
    )

    writer_result = await Runner.run(writer_agent, writer_prompt)
    report: str = writer_result.final_output

    return report

In [ ]:
# Cell 9 — Save report to disk

def save_report(topic: str, report: str) -> Path:
    """Persist the Markdown report to the reports directory."""
    slug = "_".join(topic.lower().split())[:50]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = REPORTS_DIR / f"{timestamp}_{slug}.md"
    filename.write_text(report, encoding="utf-8")
    return filename

In [ ]:
# Cell 10 — Gradio UI

import asyncio
import gradio as gr

MODELS = [
    "google/gemini-2.5-flash",
    "openai/gpt-4o",
    "anthropic/claude-sonnet-4-5",
    "deepseek/deepseek-r1",
]


def run_research(topic: str, model: str, depth: int):
    """
    Gradio handler — runs the async pipeline synchronously,
    returns (report_markdown, report_filepath) for display + download.
    """
    if not topic.strip():
        raise gr.Error("Please enter a research topic.")

    # Apply widget selections to globals before running
    global PRIMARY_MODEL, MAX_SUB_QUESTIONS
    PRIMARY_MODEL     = model
    MAX_SUB_QUESTIONS = depth

    report = asyncio.run(run_research_pipeline(topic))
    saved  = save_report(topic, report)
    return report, str(saved)


with gr.Blocks(title="Deep Research Agent", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""# Deep Research Agent
**Multi-agent pipeline:** Planner → Researcher → Writer &nbsp;|&nbsp; Powered by OpenRouter + Tavily""")

    with gr.Row():
        with gr.Column(scale=2):
            topic_box = gr.Textbox(
                label="Research Topic",
                placeholder="e.g. The geopolitical impact of sovereign AI models in 2025",
                lines=3,
            )
        with gr.Column(scale=1):
            model_dd = gr.Dropdown(
                label="Model",
                choices=MODELS,
                value=PRIMARY_MODEL,
            )
            depth_sl = gr.Slider(
                label="Research Depth (sub-questions)",
                minimum=2, maximum=8, step=1, value=MAX_SUB_QUESTIONS,
            )

    run_btn = gr.Button("Run Research", variant="primary")

    with gr.Row():
        report_md   = gr.Markdown(label="Report")
        download_file = gr.File(label="Download Report (.md)")

    run_btn.click(
        fn=run_research,
        inputs=[topic_box, model_dd, depth_sl],
        outputs=[report_md, download_file],
    )

demo.launch()
